# Décomposer la variabilité des règlements de sinistres selon la hiérarchie organisationnelle d'un assureur avec PROC NESTED

## Résumé analytique

Un assureur IARD veut savoir **où** naît l'incohérence des règlements de
sinistres : vient-elle des différences entre régions géographiques,
entre agences, entre experts individuels, ou simplement du bruit
sinistre par sinistre ? Ce notebook construit un portefeuille
synthétique et entièrement imbriqué de sinistres auto (Région > Agence
> Expert > sinistre) et utilise **PROC NESTED** pour effectuer une
analyse de variance à effets aléatoires, estimant la composante de
variance apportée par chaque niveau de la hiérarchie et l'exprimant en
pourcentage du total. Le résultat indique à la direction quel niveau
organisationnel cibler pour améliorer la cohérence des règlements.

## Sources de données

Toutes les données sont générées en ligne par la première étape DATA —
aucun fichier externe, aucun réseau. Le plan est **entièrement
imbriqué** : chaque agence appartient à exactement une région, chaque
expert à exactement une agence, et chaque sinistre à exactement un
expert.

| Jeu de données | Lignes | Variable | Type | Rôle | Description |
|---------|------|----------|------|------|-------------|
| `claims` | 40 | `Region` | car | CLASS (niveau 1, le plus externe) | Région géographique (5 régions : NE, SE, MO, CO, SO) |
| | | `Branch` | car | CLASS (niveau 2, imbriqué dans Region) | Code d'agence (2 agences par région) |
| | | `Adjuster` | car | CLASS (niveau 3, imbriqué dans Branch) | Identifiant de l'expert en sinistres (2 experts par agence) |
| | | `Settlement` | num | VAR (dépendante) | Indemnité versée sur le sinistre auto, en USD |
| | | `CycleDays` | num | VAR (dépendante) | Nombre de jours entre la première déclaration du sinistre et son règlement |

Structure synthétique : 5 régions x 2 agences x 2 experts x 2 sinistres
= 40 observations. Un effet aléatoire de région, un effet aléatoire
agence-dans-région, un effet aléatoire expert-dans-agence, et un
résidu au niveau du sinistre sont superposés de façon additive via
`rand('NORMAL', ...)` afin que les composantes de variance soient
récupérables. L'effet région est tiré avec le plus grand écart-type
(2 200) de sorte que *où* un sinistre est traité soit le facteur
dominant. Graine fixée avec `call streaminit(20260531)`. Un plan
compact et parfaitement équilibré garde chaque niveau de la hiérarchie
peuplé avec de vrais degrés de liberté.

# Décomposer la variabilité des règlements de sinistres avec PROC NESTED

Lorsqu'un assureur IARD examine ce qu'il paie pour régler les sinistres
auto, il constate souvent une grande variation. Une partie de cette
variation est inévitable (chaque accident est différent), mais une
partie reflète des facteurs **organisationnels** — une région règle
plus généreusement qu'une autre, une agence est plus généreuse, un
expert individuel est atypique.

Les données ont une structure strictement **imbriquée** (hiérarchique) :

```
Région  ->  Agence (imbriquée dans Région)  ->  Expert (imbriqué dans Agence)  ->  sinistres individuels
```

Une agence appartient à exactement une région et un expert à exactement
une agence, donc les facteurs sont imbriqués plutôt que croisés.
`PROC NESTED` effectue une analyse de variance à effets aléatoires pour
exactement ce plan et estime une **composante de variance** à chaque
niveau, exprimée en pourcentage du total. Cette répartition en
pourcentage répond à la question métier : *quel niveau de
l'organisation faut-il cibler pour rendre les règlements plus
cohérents ?*

## Étape 1 &mdash; Générer un portefeuille synthétique et entièrement imbriqué de sinistres

Nous simulons 5 régions, chacune avec 2 agences, chacune avec 2
experts, chacun traitant 2 sinistres (40 lignes au total). Nous
construisons la réponse à partir d'effets aléatoires additifs afin que
chaque niveau contribue réellement à la variance :

- un effet **région** (le plus large écart — les régions diffèrent le
  plus),
- un effet **agence-dans-région**,
- un effet **expert-dans-agence**,
- et un **résidu au niveau du sinistre** (le bruit intra-expert).

`call streaminit` fixe la graine pour la reproductibilité ; chaque
effet est tiré avec `rand('NORMAL', mean, sd)`. L'effet région utilise
l'écart-type le plus large, il devrait donc capter la plus grande part
de la variance totale. Nous générons aussi une deuxième réponse,
`CycleDays`, qui partage la même hiérarchie afin de pouvoir démontrer
plus tard l'analyse à réponses multiples.

In [1]:
DONNÉES claims;
   APPELER streaminit(20260531);
   LONGUEUR Region $2 Branch $6 Adjuster $9;

   /* Region-level random effect: regions differ the most. */
   FAIRE r = 1 JUSQU_À 5;
      Region = SCAN('NE SE MO CO SO', r);
      RegionEffect  = rand('NORMAL', 0, 2200);
      RegionCycle   = rand('NORMAL', 0, 11);

      /* Branch nested within region. */
      FAIRE b = 1 JUSQU_À 2;
         Branch = catx('-', Region, ÉCRIRE(b, z2.));
         BranchEffect = rand('NORMAL', 0, 700);
         BranchCycle  = rand('NORMAL', 0, 4);

         /* Adjuster nested within branch. */
         FAIRE a = 1 JUSQU_À 2;
            Adjuster = catx('-', Branch, ÉCRIRE(a, z1.));
            AdjEffect = rand('NORMAL', 0, 450);
            AdjCycle  = rand('NORMAL', 0, 2.5);

            /* Individual claims handled by this adjuster. */
            FAIRE claim = 1 JUSQU_À 2;
               Settlement = 8500
                          + RegionEffect + BranchEffect + AdjEffect
                          + rand('NORMAL', 0, 1100);
               CycleDays  = 21
                          + RegionCycle + BranchCycle + AdjCycle
                          + rand('NORMAL', 0, 6);
               SI Settlement < 0 ALORS Settlement = 0;
               SI CycleDays  < 1 ALORS CycleDays  = 1;
               SORTIE;
            FIN;
         FIN;
      FIN;
   FIN;

   GARDER Region Branch Adjuster Settlement CycleDays;
EXÉCUTER;



NOTE: DATA claims


NOTE: Wrote claims (40 rows, 5 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds


## Étape 2 &mdash; Trier selon la hiérarchie d'imbrication

`PROC NESTED` exige que les données en entrée soient **triées selon
les variables CLASS dans l'ordre où elles seront listées** — le
facteur le plus externe en premier. Nous trions par `Region Branch
Adjuster` afin que la procédure puisse parcourir correctement la
hiérarchie.

In [2]:
PROCÉDURE TRIER DONNÉES=claims;
   PAR Region Branch Adjuster;
EXÉCUTER;



NOTE: PROC SORT data=claims

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: Read 40 rows from claims.
NOTE: Wrote claims (40 rows, 5 columns).
NOTE: PROC SORT statement used.


## Étape 3 &mdash; Analyse des composantes de variance du montant du règlement

L'analyse centrale. Nous listons les variables CLASS du plus externe
au plus interne (`Region Branch Adjuster`) ; la réplication la plus
interne — les sinistres individuels — est automatiquement traitée
comme le terme d'erreur. `VAR Settlement` nomme l'unique réponse.

L'instruction `LABEL` fournit un nom d'affichage plus lisible pour la
réponse dans le bandeau de sortie. La sortie produit les coefficients
des carrés moyens espérés, le tableau d'analyse de variance (avec un
test F de chaque composante de variance contre zéro), et l'estimation
de la **composante de variance** à chaque niveau accompagnée de son
**pourcentage du total**.

In [3]:
TITRE 'Composantes de variance des règlements de sinistres auto';
title2 'ANOVA à effets aléatoires Région / Agence / Expert';

PROCÉDURE nested DONNÉES=claims;
   CLASSE Region Branch Adjuster;
   VAR Settlement;
   ÉTIQUETTE Region = 'Région' Branch = 'Agence' Adjuster = 'Expert'
         Settlement = 'Indemnité versée (USD)';
EXÉCUTER;


                                Composantes de variance des règlements de sinistres auto                                
                                   ANOVA à effets aléatoires Région / Agence / Expert                                   


                       Nested Random Effects Analysis of Variance

Nested Random Effects Analysis of Variance for Variable Indemnité versée (USD)
Variance Source       DF  Sum of Squares       F Value      Pr > F  Error Term   Mean Square  Variance Component    Percent of Total    EMS Coef
Région                 4  62114122.126866          6.71      0.0304    Agence  15528530.531717  1651824.844989             54.4057      8.0000
Agence                 5  11569658.859028          1.89      0.1829    Expert  2313931.771806   272682.828942              8.9813      4.0000
Expert                10  12232004.560376          1.22      0.3348     Error  1223200.456038   111582.782346              3.6752      2.0000
Error                 20  20000697.826


NOTE: Option TITLE changed to Composantes de variance des règlements de sinistres auto.
NOTE: Option TITLE2 changed to ANOVA à effets aléatoires Région / Agence / Expert.
NOTE: PROC NESTED nested ANOVA / variance components

NOTE: PROC NESTED statement used.


## Étape 4 &mdash; Analyser deux réponses à la fois

La direction se soucie aussi du **délai de traitement** — le temps que
prennent les sinistres à se régler. Nous ajoutons `CycleDays` à la
liste `VAR`. Avec plus d'une variable dépendante, `PROC NESTED` fournit
en plus une **analyse de covariation** montrant comment les deux
réponses covarient à chaque niveau de la hiérarchie (par exemple, si
les régions qui paient plus règlent aussi plus lentement).

In [4]:
TITRE 'Composantes de variance du règlement et du délai de traitement';
title2 'Deux réponses dans la même hiérarchie imbriquée';

PROCÉDURE nested DONNÉES=claims;
   CLASSE Region Branch Adjuster;
   VAR Settlement CycleDays;
   ÉTIQUETTE Region = 'Région' Branch = 'Agence' Adjuster = 'Expert'
         Settlement = 'Indemnité versée (USD)'
         CycleDays  = 'Délai de règlement (jours)';
EXÉCUTER;


                             Composantes de variance du règlement et du délai de traitement                             
                                    Deux réponses dans la même hiérarchie imbriquée                                     


                       Nested Random Effects Analysis of Variance

Nested Random Effects Analysis of Variance for Variable Indemnité versée (USD)
Variance Source       DF  Sum of Squares       F Value      Pr > F  Error Term   Mean Square  Variance Component    Percent of Total    EMS Coef
Région                 4  62114122.126866          6.71      0.0304    Agence  15528530.531717  1651824.844989             54.4057      8.0000
Agence                 5  11569658.859028          1.89      0.1829    Expert  2313931.771806   272682.828942              8.9813      4.0000
Expert                10  12232004.560376          1.22      0.3348     Error  1223200.456038   111582.782346              3.6752      2.0000
Error                 20  20000697.826


NOTE: Option TITLE changed to Composantes de variance du règlement et du délai de traitement.
NOTE: Option TITLE2 changed to Deux réponses dans la même hiérarchie imbriquée.
NOTE: PROC NESTED nested ANOVA / variance components

NOTE: PROC NESTED statement used.


## Étape 5 &mdash; Vue variance uniquement avec l'option AOV

Pour un résumé rapide des composantes de variance sur les deux
réponses, l'option `AOV` restreint la sortie aux statistiques
d'analyse de variance et **supprime la section d'analyse de
covariation**. C'est la vue compacte qu'un analyste de portefeuille
consulterait lorsqu'il n'a besoin que de la répartition de variance
par niveau pour chaque réponse, sans la covariation entre réponses.

In [5]:
TITRE 'Composantes de variance uniquement (AOV)';

PROCÉDURE nested DONNÉES=claims aov;
   CLASSE Region Branch Adjuster;
   VAR Settlement CycleDays;
   ÉTIQUETTE Region = 'Région' Branch = 'Agence' Adjuster = 'Expert'
         Settlement = 'Indemnité versée (USD)'
         CycleDays  = 'Délai de règlement (jours)';
EXÉCUTER;

TITRE;


                                        Composantes de variance uniquement (AOV)                                        
                                    Deux réponses dans la même hiérarchie imbriquée                                     


                       Nested Random Effects Analysis of Variance

Nested Random Effects Analysis of Variance for Variable Indemnité versée (USD)
Variance Source       DF  Sum of Squares       F Value      Pr > F  Error Term   Mean Square  Variance Component    Percent of Total    EMS Coef
Région                 4  62114122.126866          6.71      0.0304    Agence  15528530.531717  1651824.844989             54.4057      8.0000
Agence                 5  11569658.859028          1.89      0.1829    Expert  2313931.771806   272682.828942              8.9813      4.0000
Expert                10  12232004.560376          1.22      0.3348     Error  1223200.456038   111582.782346              3.6752      2.0000
Error                 20  20000697.826


NOTE: Option TITLE changed to Composantes de variance uniquement (AOV).
NOTE: PROC NESTED nested ANOVA / variance components

NOTE: PROC NESTED statement used.


## Interprétation des résultats

La colonne **Pourcentage du total** du tableau d'analyse de variance
est l'information clé. Lue de haut en bas, elle donne la part de la
variabilité totale des règlements attribuable à chaque niveau
organisationnel. Pour le montant du règlement, l'exécution produit :

- **Région — 54,4 %** (Pr > F = 0,0304). Les données ont été générées
  avec le plus grand écart au niveau région, et l'analyse le retrouve :
  plus de la moitié de toute la variabilité des règlements se situe
  *entre* les régions, une preuve statistiquement significative que
  *où* un sinistre est traité est le facteur dominant.
- **Agence (dans Région) — 9,0 %** (Pr > F = 0,18). Une part
  supplémentaire modeste une fois qu'on descend de la région aux
  agences individuelles ; non significative ici.
- **Expert (dans Agence) — 3,7 %** (Pr > F = 0,33). Peu de variation
  est imputable à l'expert individuel dans ce portefeuille.
- **Erreur — 32,9 %.** Le bruit résiduel sinistre par sinistre au sein
  d'un expert ; c'est la variation irréductible qu'aucun levier
  organisationnel ne peut supprimer.

Chaque niveau porte aussi un **test F (Pr > F)** de l'hypothèse nulle
selon laquelle sa composante de variance est nulle. La faible valeur p
de la Région (0,0304) est une preuve statistique de différences
systématiques réelles entre régions, et non du hasard d'échantillonnage.

La réponse délai de traitement raconte une histoire parallèle : **la
Région explique 45,8 %** de la variation du délai de règlement
(Pr > F = 0,0448, à nouveau significatif), l'Agence et l'Expert
contribuant chacun pour une part à un chiffre, et le résidu portant
40,1 %. Ainsi, tant *combien* un assureur paie que *combien de temps*
cela prend sont d'abord et avant tout gouvernés par la région.

L'exécution à deux réponses ajoute une **analyse de covariation**. Ici,
le niveau Région domine les produits croisés, et le **coefficient de
corrélation global est de -0,36** — une relation négative : les
régions qui paient des règlements plus élevés ont tendance à les
*régler plus vite*, pas plus lentement. C'est une constatation utile et
non évidente — les régions coûteuses ne sont pas les plus lentes.

**Conclusion pour le métier :** comme la Région domine la répartition
en pourcentage du total pour les deux réponses, l'assureur devrait
d'abord uniformiser les directives de règlement et les limites de
délégation *entre régions* — c'est là que se situe l'incohérence la
plus importante et statistiquement significative — avant d'investir
dans des interventions au niveau des agences ou des experts. La
corrélation négative entre règlement et délai de traitement signifie
qu'aucune région n'est à la fois la plus coûteuse et la plus lente, si
bien que les deux problèmes peuvent être traités par des programmes
distincts, ciblés par région. PROC NESTED transforme le sentiment vague
que « nos règlements sont incohérents » en une attribution
actionnable, niveau par niveau, de l'origine de cette incohérence.